[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/D.%20Integrated%20Tactical%20%26%20Pre-Planning%20Problems%20%28Connecting%20the%20Silos%29/34.%20The%20Hazard%20%26%20IMDG%20Segregation%20Problem/P34-Tier-1_executed.ipynb)

---


# 34. The Hazard and IMDG Segregation Problem

## Tier 1: The Pen-and-Paper Method (Mixed-Integer Programming Formulation)

### Goal

Formulate and solve a small IMDG segregation problem as a mixed-integer programming (MIP) model. The model assigns each container to a stowage position, penalizes any modeled segregation violation, and prefers positions in lower-numbered holds when other considerations are equal.

A MIP is an optimization model with a linear objective and linear constraints, in which some or all decision variables must take integer values. Here, binary variables represent yes-or-no decisions: whether a container uses a position and whether a container pair incurs a violation penalty. The solver searches for variable values that satisfy the constraints and minimize the objective.

### Inputs, outputs, and scope

- **Inputs:** containers and their IMDG classes, available positions and their holds and coordinates, and pairwise segregation requirements.
- **Outputs:** a solver status, an objective value when a feasible solution is available, container-to-position assignments, and container pairs marked as violations by the model.
- **Scope:** this is a small teaching instance. It does not represent every operational, structural, loading, or regulatory detail of a vessel, and its simplified geometry is not a substitute for an operational IMDG assessment.

### Model assumptions

- Each container is assigned to exactly one position; each position holds at most one container.
- Coordinates are local to a hold. Euclidean distances are calculated in three dimensions between positions in the same hold. For this example, positions in different holds are treated as satisfying an "Away From" distance requirement; a different-hold requirement is still checked explicitly. The layout graphic shows the horizontal and vertical coordinates, so it is a two-dimensional projection of the positions.
- The position-preference cost is the hold number, so a lower-numbered hold has a lower cost. This is a simple preference, not a physical stability model. Container weights are recorded in the data but are not used in this objective.
- A binary violation variable represents a container pair's violation penalty. The penalty is deliberately large relative to the total possible position-preference cost, so a violation-free assignment is preferred whenever one exists.

### Approach

1. Define the data structures and the concrete example.
2. Calculate within-hold distances and explain how segregation is checked.
3. Formulate the assignment and violation decisions as a MIP.
4. Solve the model, verify the assignments, and visualize the result.
5. Examine how changing the minimum-distance requirements affects the model.

### Concrete example

The example has four containers and six positions distributed across two holds:

| Container | IMDG class | Example description |
|---|---|---|
| A | 1.4S | Explosives |
| B | 3 | Flammable liquid |
| C | 8.1 | Corrosive acid |
| D | 2.1 | Flammable gas |

The pairwise requirements are: A–B, A–D, B–C, and C–D must be "Away From" by at least 3 m; A–C must be in different holds; and B–D has no restriction. The code below is the authoritative specification of this example.

In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Tuple
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pulp
import seaborn as sns
from matplotlib.patches import Patch

# A consistent plotting style makes the assignments and compliance results easier to read.
plt.style.use("default")
sns.set_palette("husl")

In [ ]:
@dataclass
class Container:
    """A container carrying dangerous goods."""

    id: str
    imdg_class: str
    weight: float = 20.0  # Nominal weight in metric tonnes; unused by this model.


@dataclass
class Position:
    """A stowage position identified by its hold and local coordinates."""

    id: str
    hold: int
    x: float  # Horizontal coordinate in meters, local to the hold.
    y: float  # Vertical coordinate in meters, local to the hold.
    z: float  # Depth coordinate in meters, local to the hold.


@dataclass
class SegregationRequirement:
    """A pairwise IMDG-class requirement used by the example model."""

    class1: str
    class2: str
    requirement: str  # "Away From", "Separated From", or "No restriction".
    min_distance: float  # Minimum distance in meters when applicable.
    different_hold: bool = False

In [ ]:
def create_concrete_example():
    """Create the example with four containers, six positions, and six requirements."""

    containers = [
        Container("A", "1.4S"),  # Explosives.
        Container("B", "3"),     # Flammable liquid.
        Container("C", "8.1"),   # Corrosive acid.
        Container("D", "2.1"),   # Flammable gas.
    ]

    # Each hold has three local positions at 3 m intervals.
    positions = [
        Position("H1_P1", 1, 0, 0, 0),
        Position("H1_P2", 1, 3, 0, 0),
        Position("H1_P3", 1, 6, 0, 0),
        Position("H2_P1", 2, 0, 0, 0),
        Position("H2_P2", 2, 3, 0, 0),
        Position("H2_P3", 2, 6, 0, 0),
    ]

    segregation_requirements = [
        SegregationRequirement("1.4S", "3", "Away From", 3.0),
        SegregationRequirement("1.4S", "8.1", "Separated From", 0.0, True),
        SegregationRequirement("1.4S", "2.1", "Away From", 3.0),
        SegregationRequirement("3", "8.1", "Away From", 3.0),
        SegregationRequirement("3", "2.1", "No restriction", 0.0),
        SegregationRequirement("8.1", "2.1", "Away From", 3.0),
    ]

    return containers, positions, segregation_requirements


containers, positions, seg_reqs = create_concrete_example()

print("Containers:")
for container in containers:
    print(f"  {container.id}: Class {container.imdg_class}")

print("\nPositions:")
for position in positions:
    print(
        f"  {position.id}: Hold {position.hold}, "
        f"local position ({position.x}, {position.y}, {position.z}) m"
    )

print("\nSegregation requirements:")
for requirement in seg_reqs:
    print(
        f"  {requirement.class1} - {requirement.class2}: "
        f"{requirement.requirement} "
        f"(minimum: {requirement.min_distance} m, "
        f"different holds: {requirement.different_hold})"
    )

In [ ]:
def calculate_distance_matrix(
    positions: List[Position],
) -> Dict[Tuple[str, str], float]:
    """Calculate Euclidean distances for every pair of positions in one hold."""

    distances = {}
    for pos1 in positions:
        for pos2 in positions:
            if pos1.hold == pos2.hold:
                dx = pos1.x - pos2.x
                dy = pos1.y - pos2.y
                dz = pos1.z - pos2.z
                distances[(pos1.id, pos2.id)] = math.sqrt(
                    dx**2 + dy**2 + dz**2
                )
    return distances


def check_segregation(
    container1: Container,
    container2: Container,
    pos1: Position,
    pos2: Position,
    requirements: List[SegregationRequirement],
    distances: Dict[Tuple[str, str], float],
) -> bool:
    """Return whether this container pair satisfies its applicable requirement."""

    applicable_req = None
    for req in requirements:
        classes_match = (
            req.class1 == container1.imdg_class
            and req.class2 == container2.imdg_class
        ) or (
            req.class1 == container2.imdg_class
            and req.class2 == container1.imdg_class
        )
        if classes_match:
            applicable_req = req
            break

    if applicable_req is None or applicable_req.requirement == "No restriction":
        return True

    if applicable_req.different_hold and pos1.hold == pos2.hold:
        return False

    # Different holds satisfy the example's distance rule by assumption; their
    # local coordinates do not define a meaningful cross-hold Euclidean distance.
    if pos1.hold != pos2.hold:
        return True

    distance = distances[(pos1.id, pos2.id)]
    return distance >= applicable_req.min_distance


distance_matrix = calculate_distance_matrix(positions)

print("Within-hold distance matrix entries (meters):")
for (pos1_id, pos2_id), distance in distance_matrix.items():
    if pos1_id != pos2_id:
        print(f"  {pos1_id} - {pos2_id}: {distance:.1f} m")
print("Positions in different holds are handled by the stated model assumption.")

## MIP model

### Intuition and decisions

The model must choose one position for each container while preventing incompatible pairs from occupying prohibited position combinations. A binary assignment variable records each placement. A second binary variable, indexed by an unordered container pair, records whether that pair incurs a segregation-violation penalty. Indexing by an unordered pair avoids charging separately for the same pair in the two possible orders.

A position pair is prohibited when it violates the applicable rule: for a positive minimum-distance requirement, both positions are in the same hold and their within-hold distance is too small; for a different-hold requirement, both positions are in the same hold. Different holds satisfy an "Away From" distance rule under the stated assumption. A pair with no applicable rule or with "No restriction" contributes no segregation constraints.

Let $I$ be the set of containers and $P$ the set of positions. For each unordered container pair $i<j$, let $F_{ij} \subseteq P \times P$ be the set of prohibited ordered position pairs for that pair's requirement. In the implementation, these combinations are identified by checking position holds and distances before the corresponding constraints are added.

Define binary variables $x_{ip}$ and $y_{ij}$:

- $x_{ip}=1$ if container $i$ is assigned to position $p$, and $0$ otherwise.
- $y_{ij}=1$ if the model incurs a violation penalty for container pair $(i,j)$, and $0$ otherwise. A prohibited placement forces $y_{ij}=1$; because the objective penalizes $y_{ij}$, the solver sets it to zero when no prohibited placement for that pair is selected in an optimal solution.

The model minimizes violation penalties plus position-preference costs:

$$
\min \quad M \sum_{i<j} y_{ij} + \sum_{i \in I}\sum_{p \in P} w_p x_{ip}
$$

subject to the assignment and capacity constraints:

$$
\sum_{p \in P} x_{ip} = 1 \quad \forall i \in I,
\qquad
\sum_{i \in I} x_{ip} \leq 1 \quad \forall p \in P,
$$

and the segregation constraints:

$$
x_{ip} + x_{jq} \leq 1 + y_{ij}
\quad \forall i<j,\ (p,q)\in F_{ij},
$$

with

$$
x_{ip}, y_{ij} \in \{0,1\}.
$$

The first constraint assigns every container to exactly one position. The second enforces the capacity of one container per position. In the segregation constraint, selecting both prohibited positions makes the left-hand side equal to 2, which requires $y_{ij}=1$. If at least one of those positions is not selected, the constraint does not force a violation penalty.

Here, $M=10000$ is the violation penalty and $w_p$ is the hold number of position $p$. With four containers and hold numbers no greater than 2, the total position-preference cost is at most 8. Thus, the penalty makes a violation-free assignment preferable whenever one exists. When no violation-free assignment exists, the objective prioritizes reducing the number of penalized pairs before minimizing hold-number costs. The model does not impose a separate hard prohibition on violations.

In the code, $x_{ip}$ is stored as `x[(i, p)]`, $y_{ij}$ as `y[(i, j)]`, $M$ as `M`, and $w_p$ is obtained from `position.hold`. `F_{ij}` is constructed directly in the segregation-constraint loops. The binary restrictions are implemented as integer variables with lower bound 0 and upper bound 1, so each variable can take only the values 0 and 1.

The model has $|I||P| + |I|(|I|-1)/2$ binary variables. At most $|I|(|I|-1)|P|^2/2$ segregation constraints can be generated, since each unordered container pair can be checked against every ordered pair of positions. These counts describe the formulation's growth; the time required by a MIP solver can grow substantially as the instance becomes larger.

In [ ]:
def solve_imdg_segregation_mip(
    containers: List[Container],
    positions: List[Position],
    requirements: List[SegregationRequirement],
    distances: Dict[Tuple[str, str], float],
) -> Dict:
    """Solve the IMDG segregation assignment model as a mixed-integer program."""

    prob = pulp.LpProblem("IMDG_Segregation", pulp.LpMinimize)

    # Positional bounds keep binary-variable construction compatible with PuLP APIs
    # whose parameter names differ, while preserving the same 0-or-1 integer domain.
    x = {
        (i, p): pulp.LpVariable(
            f"x_{i}_{p}", 0, 1, pulp.LpInteger
        )
        for i in range(len(containers))
        for p in range(len(positions))
    }

    # One violation variable per unordered container pair avoids duplicate penalties.
    y = {
        (i, j): pulp.LpVariable(
            f"y_{i}_{j}", 0, 1, pulp.LpInteger
        )
        for i in range(len(containers))
        for j in range(i + 1, len(containers))
    }

    M = 10000
    position_weights = {p: position.hold for p, position in enumerate(positions)}

    prob += (
        pulp.lpSum(M * y[(i, j)] for i, j in y)
        + pulp.lpSum(
            position_weights[p] * x[(i, p)]
            for i in range(len(containers))
            for p in range(len(positions))
        ),
        "Total_Cost",
    )

    # Every container receives one position, and each position has capacity one.
    for i in range(len(containers)):
        prob += (
            pulp.lpSum(x[(i, p)] for p in range(len(positions))) == 1,
            f"Container_{i}_Assignment",
        )

    for p in range(len(positions)):
        prob += (
            pulp.lpSum(x[(i, p)] for i in range(len(containers))) <= 1,
            f"Position_{p}_Capacity",
        )

    # Prohibited placement pairs are represented with linear binary constraints.
    for i in range(len(containers)):
        for j in range(i + 1, len(containers)):
            container1, container2 = containers[i], containers[j]
            applicable_req = None

            for req in requirements:
                classes_match = (
                    req.class1 == container1.imdg_class
                    and req.class2 == container2.imdg_class
                ) or (
                    req.class1 == container2.imdg_class
                    and req.class2 == container1.imdg_class
                )
                if classes_match:
                    applicable_req = req
                    break

            if applicable_req is None or applicable_req.requirement == "No restriction":
                continue

            for p1, pos1 in enumerate(positions):
                for p2, pos2 in enumerate(positions):
                    violates_distance = (
                        applicable_req.min_distance > 0
                        and pos1.hold == pos2.hold
                        and distances[(pos1.id, pos2.id)] < applicable_req.min_distance
                    )
                    violates_hold = (
                        applicable_req.different_hold
                        and pos1.hold == pos2.hold
                    )

                    if violates_distance or violates_hold:
                        prob += (
                            x[(i, p1)] + x[(j, p2)] <= 1 + y[(i, j)],
                            f"Segregation_{i}_{j}_{p1}_{p2}",
                        )

    print("Solving the MIP optimization problem...")
    prob.solve(pulp.PULP_CBC_CMD(msg=False))

    status = pulp.LpStatus[prob.status]
    objective_value = pulp.value(prob.objective)
    assignments = {}
    violations = []

    if status in {"Optimal", "Feasible"}:
        for i, container in enumerate(containers):
            for p, position in enumerate(positions):
                value = pulp.value(x[(i, p)])
                if value is not None and value > 0.5:
                    assignments[container.id] = position
                    break

        for (i, j), variable in y.items():
            value = pulp.value(variable)
            if value is not None and value > 0.5:
                violations.append((containers[i].id, containers[j].id))

    return {
        "status": status,
        "objective_value": objective_value,
        "assignments": assignments,
        "violations": violations,
        "problem": prob,
    }

In [ ]:
result = solve_imdg_segregation_mip(
    containers, positions, seg_reqs, distance_matrix
)

print(f"Optimization status: {result['status']}")
if result["objective_value"] is not None:
    print(f"Objective value: {result['objective_value']:.2f}")
print(f"Number of modeled violating pairs: {len(result['violations'])}")

if result["status"] in {"Optimal", "Feasible"}:
    if result["violations"]:
        print("Pairs marked as segregation violations:")
        for container1_id, container2_id in result["violations"]:
            print(f"  {container1_id} - {container2_id}")
    else:
        print("No segregation violations are needed by the model.")

    print("\nContainer assignments:")
    for container_id, position in result["assignments"].items():
        container = next(c for c in containers if c.id == container_id)
        print(
            f"  {container_id} (Class {container.imdg_class}) -> "
            f"{position.id} (Hold {position.hold})"
        )
else:
    print("The solver did not return a feasible assignment for this model.")

In [ ]:
def verify_solution(
    assignments: Dict[str, Position],
    containers: List[Container],
    requirements: List[SegregationRequirement],
    distances: Dict[Tuple[str, str], float],
) -> List[Dict]:
    """Check each container pair against the stated segregation rules."""

    verification_results = []
    for i, container1 in enumerate(containers):
        for j, container2 in enumerate(containers):
            if i >= j:
                continue

            pos1 = assignments[container1.id]
            pos2 = assignments[container2.id]
            applicable_req = None

            for req in requirements:
                classes_match = (
                    req.class1 == container1.imdg_class
                    and req.class2 == container2.imdg_class
                ) or (
                    req.class1 == container2.imdg_class
                    and req.class2 == container1.imdg_class
                )
                if classes_match:
                    applicable_req = req
                    break

            if applicable_req is None:
                continue

            is_compliant = check_segregation(
                container1, container2, pos1, pos2, requirements, distances
            )
            actual_distance = (
                distances[(pos1.id, pos2.id)]
                if pos1.hold == pos2.hold
                else None
            )

            verification_results.append(
                {
                    "container1": container1.id,
                    "class1": container1.imdg_class,
                    "container2": container2.id,
                    "class2": container2.imdg_class,
                    "requirement": applicable_req.requirement,
                    "min_distance": applicable_req.min_distance,
                    "different_hold": applicable_req.different_hold,
                    "pos1": pos1.id,
                    "pos2": pos2.id,
                    "actual_distance": actual_distance,
                    "pos1_hold": pos1.hold,
                    "pos2_hold": pos2.hold,
                    "compliant": is_compliant,
                }
            )

    return verification_results


if result["status"] in {"Optimal", "Feasible"}:
    verification = verify_solution(
        result["assignments"], containers, seg_reqs, distance_matrix
    )

    assert len(result["assignments"]) == len(containers)
    assert len({position.id for position in result["assignments"].values()}) == len(
        containers
    )

    verified_violations = {
        (row["container1"], row["container2"])
        for row in verification
        if not row["compliant"]
    }
    assert verified_violations == set(result["violations"])

    print("Solution verification:")
    print("=" * 72)
    for row in verification:
        status_symbol = "PASS" if row["compliant"] else "FAIL"
        print(
            f"{status_symbol}: {row['container1']} ({row['class1']}) - "
            f"{row['container2']} ({row['class2']}): {row['requirement']}"
        )
        print(
            f"  Positions: {row['pos1']} (Hold {row['pos1_hold']}) - "
            f"{row['pos2']} (Hold {row['pos2_hold']})"
        )
        if row["actual_distance"] is None:
            print("  Distance: across different holds (treated as separated)")
        else:
            print(
                f"  Distance: {row['actual_distance']:.1f} m "
                f"(minimum: {row['min_distance']:.1f} m)"
            )
        if row["different_hold"]:
            print(
                "  Different-hold requirement: "
                f"{'satisfied' if row['pos1_hold'] != row['pos2_hold'] else 'not satisfied'}"
            )
        print()

In [ ]:
def visualize_solution(
    assignments: Dict[str, Position],
    containers: List[Container],
    positions: List[Position],
    verification: List[Dict],
) -> None:
    """Plot the hold layouts and summarize pairwise segregation compliance."""

    container_by_id = {container.id: container for container in containers}

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    for ax, hold in zip(axes, (1, 2)):
        hold_positions = [position for position in positions if position.hold == hold]
        for position in hold_positions:
            assigned_container_id = next(
                (
                    container_id
                    for container_id, assigned_position in assignments.items()
                    if assigned_position.id == position.id
                ),
                None,
            )

            if assigned_container_id is None:
                ax.scatter(
                    position.x, position.y, s=500, c="lightgray", marker="s"
                )
                ax.text(
                    position.x, position.y, "Empty", ha="center", va="center", fontsize=9
                )
            else:
                container = container_by_id[assigned_container_id]
                ax.scatter(position.x, position.y, s=500, c="red", marker="s", alpha=0.7)
                ax.text(
                    position.x,
                    position.y,
                    f"{container.id}\nClass {container.imdg_class}",
                    ha="center",
                    va="center",
                    fontsize=9,
                    fontweight="bold",
                )

        ax.set_title(f"Hold {hold} layout (horizontal-vertical projection)")
        ax.set_xlabel("Horizontal position within hold (m)")
        ax.set_ylabel("Vertical position within hold (m)")
        ax.grid(True, alpha=0.3)
        ax.set_aspect("equal", adjustable="datalim")
        ax.legend(
            handles=[
                Patch(facecolor="red", alpha=0.7, label="Occupied"),
                Patch(facecolor="lightgray", label="Empty"),
            ],
            loc="best",
        )

    fig.suptitle("IMDG segregation: container placement by hold")
    fig.tight_layout()
    plt.show()

    fig, ax = plt.subplots(figsize=(12, 7))
    ax.axis("off")
    table_data = [
        ["Container pair", "IMDG classes", "Requirement", "Positions", "Distance", "Compliance"]
    ]
    cell_colours = [["#4472C4"] * 6]

    for row in verification:
        distance_text = (
            "Across holds"
            if row["actual_distance"] is None
            else f"{row['actual_distance']:.1f} m"
        )
        table_data.append(
            [
                f"{row['container1']} - {row['container2']}",
                f"{row['class1']} - {row['class2']}",
                row["requirement"],
                f"{row['pos1']} - {row['pos2']}",
                distance_text,
                "PASS" if row["compliant"] else "FAIL",
            ]
        )
        compliance_colour = "lightgreen" if row["compliant"] else "lightcoral"
        cell_colours.append(["white"] * 5 + [compliance_colour])

    table = ax.table(
        cellText=table_data,
        cellColours=cell_colours,
        cellLoc="center",
        loc="center",
        colWidths=[0.15, 0.15, 0.19, 0.21, 0.12, 0.12],
    )
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1, 1.8)
    for column in range(6):
        table[(0, column)].set_text_props(weight="bold", color="white")

    ax.set_title("IMDG segregation compliance by container pair", pad=18)
    plt.show()


if result["status"] in {"Optimal", "Feasible"}:
    visualize_solution(
        result["assignments"], containers, positions, verification
    )

In [ ]:
def sensitivity_analysis(
    containers: List[Container],
    positions: List[Position],
    requirements: List[SegregationRequirement],
    distances: Dict[Tuple[str, str], float],
) -> None:
    """Solve the same MIP under scaled minimum-distance requirements."""

    distance_multipliers = [0.5, 1.0, 1.5, 2.0]
    results = []

    for multiplier in distance_multipliers:
        modified_reqs = [
            SegregationRequirement(
                req.class1,
                req.class2,
                req.requirement,
                req.min_distance * multiplier,
                req.different_hold,
            )
            for req in requirements
        ]

        modified_result = solve_imdg_segregation_mip(
            containers, positions, modified_reqs, distances
        )
        has_solution = modified_result["status"] in {"Optimal", "Feasible"}
        results.append(
            {
                "multiplier": multiplier,
                "objective": modified_result["objective_value"] if has_solution else None,
                "violations": len(modified_result["violations"]) if has_solution else None,
                "status": modified_result["status"],
            }
        )

    multipliers = [row["multiplier"] for row in results]
    objectives = [
        row["objective"] if row["objective"] is not None else np.nan
        for row in results
    ]
    violation_counts = [
        row["violations"] if row["violations"] is not None else np.nan
        for row in results
    ]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    ax1.plot(multipliers, objectives, "bo-", linewidth=2, markersize=8)
    ax1.set_xlabel("Minimum-distance multiplier")
    ax1.set_ylabel("Objective value (model cost)")
    ax1.set_title("Objective value under scaled distance requirements")
    ax1.set_xticks(multipliers)
    ax1.grid(True, alpha=0.3)

    bar_colours = [
        "green" if value == 0 else "red" if not np.isnan(value) else "gray"
        for value in violation_counts
    ]
    ax2.bar(range(len(multipliers)), violation_counts, color=bar_colours)
    ax2.set_xlabel("Minimum-distance multiplier")
    ax2.set_ylabel("Number of modeled violating pairs")
    ax2.set_title("Violating pairs under scaled distance requirements")
    ax2.set_xticks(range(len(multipliers)))
    ax2.set_xticklabels([f"{multiplier:g}x" for multiplier in multipliers])
    ax2.grid(True, alpha=0.3, axis="y")

    for index, value in enumerate(violation_counts):
        if not np.isnan(value):
            ax2.text(index, value, str(int(value)), ha="center", va="bottom")

    fig.suptitle("Sensitivity analysis: minimum-distance requirements")
    fig.tight_layout()
    plt.show()

    print("Sensitivity analysis results:")
    print("=" * 56)
    for row in results:
        print(f"Distance multiplier: {row['multiplier']:g}x")
        if row["objective"] is None:
            print("  Objective value: not available (no feasible solution)")
            print("  Violating pairs: not available")
        else:
            print(f"  Objective value: {row['objective']:.2f}")
            print(f"  Modeled violating pairs: {row['violations']}")
        print(f"  Solver status: {row['status']}\n")


sensitivity_analysis(containers, positions, seg_reqs, distance_matrix)

## Interpreting the method

A MIP expresses discrete choices—here, container-to-position assignments—as integer-valued variables and uses linear constraints to describe permitted combinations. The solver searches for an assignment minimizing the stated objective. An `Optimal` status means optimal for this mathematical model; it does not establish that the simplified assumptions capture every operational or regulatory consideration. A non-optimal status should not be interpreted as proof that the returned assignments are optimal.

The objective combines two different ideas: the large penalty discourages modeled segregation violations, while the hold-number cost ranks otherwise comparable assignments. The verification step checks the returned placements against the pairwise rules independently of the optimization constraints and compares the violating pairs it finds with those marked by the model. A zero violation count is meaningful only with the stated data and modeling assumptions. Different assignments can have the same objective value, so the solver may return one of several optimal placements.

The sensitivity analysis keeps the model structure and example data fixed while scaling the minimum-distance requirements. Its objective and violation-count plots show how the model responds to those changed requirements; they do not represent changes to the underlying vessel or to the regulatory rules themselves.

## Strengths and limitations

**Strengths**

- The assignment and segregation logic is explicit and auditable.
- Integer variables with bounds of zero and one represent discrete placements and pairwise violations directly.
- An optimal solver status provides an optimality guarantee for the encoded model.
- The formulation provides a rigorous small-instance baseline for mathematical study.

**Limitations**

- The number of assignment variables and pairwise placement constraints grows with the number of containers and positions, which can make larger instances computationally demanding.
- The result depends on the supplied segregation rules, geometry, and hold assumptions.
- The hold-number preference is a simplified cost and does not model vessel stability or operational handling.
- Real applications may require additional vessel, cargo, access, and regulatory constraints.

## When this formulation is useful

This MIP is useful for small instances where the assignment logic should be examined transparently, for studying the effects of explicit constraints, and for establishing a mathematical benchmark. Applying it to operational stowage requires validated data and a model that represents the relevant vessel geometry and regulations.